# PDBsum-style Interface Analyzer (Colab, simplified)

A no-install-hassle, Biopython-free replacement for the parts of the PDBsum web server
that generate ligand-plots (H-bonds / salt bridges / hydrophobic contacts), a secondary
structure track, and a Ramachandran plot -- useful when the PDBsum server itself is down.

**Two ways to use this notebook:**
- **Section A** -- analyze **one** complex (upload your own `.pdb`, or fetch a PDB ID from RCSB).
- **Section B** -- **batch** analyze every `.pdb` file in a folder you upload (e.g. a zip of docking results).

Run the **Setup** cell first either way.


In [ ]:
#@title Setup: install deps and get the analyzer code { display-mode: "form" }
!pip -q install numpy matplotlib

import os, sys, glob, shutil, json as _json

REPO_URL = "https://github.com/zaaraishaq/PDBsum-results-visualization"
REPO_DIR = "/content/PDBsum-results-visualization"

# Try to grab the analyzer straight from the GitHub repo. If it hasn't been
# pushed there yet, fall back to asking you to upload the .py file manually.
got_it = False
if not os.path.exists(REPO_DIR):
    ret = os.system(f"git clone -q {REPO_URL}.git {REPO_DIR}")
    got_it = (ret == 0) and os.path.exists(os.path.join(REPO_DIR, "src", "pdb_interface_analyzer.py"))

if got_it:
    sys.path.insert(0, os.path.join(REPO_DIR, "src"))
    print("Loaded pdb_interface_analyzer.py from the GitHub repo.")
else:
    print("Could not clone/find the script in the repo yet.")
    print("Please upload 'pdb_interface_analyzer.py' now:")
    from google.colab import files
    uploaded = files.upload()
    assert "pdb_interface_analyzer.py" in uploaded, "You must upload pdb_interface_analyzer.py"
    sys.path.insert(0, "/content")

import pdb_interface_analyzer as pia
print("Ready.")


## Section A -- analyze a single complex

Either:
1. Run the next cell and upload your own `.pdb` file, **or**
2. Skip the upload and instead set `PDB_ID` below to fetch a structure straight from RCSB
   (e.g. `"1YCR"`, the MDM2/p53 example used in this repo).


In [ ]:
#@title A1. Get a PDB file { display-mode: "form" }
PDB_ID = "1YCR"  #@param {type:"string"}
USE_RCSB_FETCH = True  #@param {type:"boolean"}

pdb_path = None

if not USE_RCSB_FETCH:
    from google.colab import files
    uploaded = files.upload()
    pdb_path = list(uploaded.keys())[0]
else:
    import urllib.request
    pdb_path = f"/content/{PDB_ID}.pdb"
    url = f"https://files.rcsb.org/download/{PDB_ID}.pdb"
    urllib.request.urlretrieve(url, pdb_path)
    print(f"Downloaded {PDB_ID} to {pdb_path}")


In [ ]:
#@title A2. Set chains and run { display-mode: "form" }
LIGAND_CHAIN = "B"  #@param {type:"string"}
RECEPTOR_CHAIN = "A"  #@param {type:"string"}
LIGAND_LABEL = "Peptide"  #@param {type:"string"}
RECEPTOR_LABEL = "Target"  #@param {type:"string"}
AUTO_DETECT_CHAINS = False  #@param {type:"boolean"}

out_dir = "/content/results_single"
os.makedirs(out_dir, exist_ok=True)

analyzer = pia.analyze_one_complex(
    pdb_path,
    receptor_chains=None if AUTO_DETECT_CHAINS else [RECEPTOR_CHAIN],
    ligand_chains=None if AUTO_DETECT_CHAINS else [LIGAND_CHAIN],
    output_root=out_dir,
    ligand_label=LIGAND_LABEL,
    receptor_label=RECEPTOR_LABEL,
)
assert analyzer is not None, "Analysis failed -- check the chain IDs above."
result_dir = os.path.join(out_dir, analyzer.complex_label)
print("Results folder:", result_dir)


In [ ]:
#@title A3. Show the text report { display-mode: "form" }
with open(os.path.join(result_dir, "Complex_Interface_Report.txt")) as f:
    print(f.read())


In [ ]:
#@title A4. Show all figures inline { display-mode: "form" }
from IPython.display import Image, display

for fname in ["interface_summary_bubble.png", "interface_interactions_network.png",
              "interface_contact_heatmap.png", "docked_complex_3d_static.png",
              "secondary_structure.png", "ramachandran_plot.png"]:
    fpath = os.path.join(result_dir, fname)
    if os.path.exists(fpath):
        print(fname)
        display(Image(filename=fpath))


In [ ]:
#@title A5. Ramachandran summary text { display-mode: "form" }
rama_path = os.path.join(result_dir, "Ramachandran_Summary.txt")
if os.path.exists(rama_path):
    with open(rama_path) as f:
        print(f.read())


In [ ]:
#@title A6. (optional) Open the interactive 3D viewer inline { display-mode: "form" }
from IPython.display import IFrame
html_path = os.path.join(result_dir, "docked_complex_3d.html")
IFrame(src=html_path, width=900, height=650)


In [ ]:
#@title A7. Download all results as a zip { display-mode: "form" }
import shutil
zip_path = shutil.make_archive("/content/single_complex_results", "zip", result_dir)
from google.colab import files
files.download(zip_path)


---
## Section B -- batch mode (multiple complexes at once)

Upload several `.pdb` files (or a `.zip` of them) and this section will run the full
analysis -- including secondary structure and Ramachandran plots -- on every one, plus
a combined `Summary_All_Complexes.txt` comparing them.

If your files don't all share the same chain IDs, edit `CHAIN_MAP` below (JSON:
`{"filename.pdb": {"ligand": "A", "receptor": "W"}, ...}`) -- any file left out of the
map will have its ligand/receptor chains auto-detected by size.


In [ ]:
#@title B1. Upload PDB files (multiple .pdb, or one .zip of them) { display-mode: "form" }
from google.colab import files
import zipfile

batch_dir = "/content/batch_input"
os.makedirs(batch_dir, exist_ok=True)

uploaded = files.upload()
for fname, data in uploaded.items():
    fpath = os.path.join(batch_dir, fname)
    with open(fpath, "wb") as f:
        f.write(data)
    if fname.lower().endswith(".zip"):
        with zipfile.ZipFile(fpath) as z:
            z.extractall(batch_dir)

pdb_files = glob.glob(os.path.join(batch_dir, "**", "*.pdb"), recursive=True)
print(f"Found {len(pdb_files)} .pdb file(s):")
for p in pdb_files:
    print(" -", p)


In [ ]:
#@title B2. (optional) Per-file chain map -- leave empty dict to auto-detect for all { display-mode: "form" }
CHAIN_MAP = {}  # e.g. {"complex1.pdb": {"ligand": "A", "receptor": "W"}}

LIGAND_LABEL = "Ligand"  #@param {type:"string"}
RECEPTOR_LABEL = "Receptor"  #@param {type:"string"}


In [ ]:
#@title B3. Run batch analysis on everything found above { display-mode: "form" }
batch_out = "/content/results_batch"
os.makedirs(batch_out, exist_ok=True)

results = []
for pdb_path in pdb_files:
    fname = os.path.basename(pdb_path)
    entry = CHAIN_MAP.get(fname, {})
    ligand_chains = [entry["ligand"]] if "ligand" in entry else None
    receptor_chains = [entry["receptor"]] if "receptor" in entry else None
    a = pia.analyze_one_complex(pdb_path, receptor_chains, ligand_chains, batch_out,
                                 ligand_label=LIGAND_LABEL, receptor_label=RECEPTOR_LABEL)
    results.append(a)

pia.write_summary_across_complexes(results, batch_out)
with open(os.path.join(batch_out, "Summary_All_Complexes.txt")) as f:
    print(f.read())


In [ ]:
#@title B4. Preview figures for one complex from the batch { display-mode: "form" }
COMPLEX_TO_PREVIEW = ""  #@param {type:"string"}

from IPython.display import Image, display
target = COMPLEX_TO_PREVIEW or (results[0].complex_label if results and results[0] else "")
preview_dir = os.path.join(batch_out, target)
if os.path.isdir(preview_dir):
    for fname in ["interface_summary_bubble.png", "secondary_structure.png", "ramachandran_plot.png"]:
        fpath = os.path.join(preview_dir, fname)
        if os.path.exists(fpath):
            print(fname)
            display(Image(filename=fpath))
else:
    print("Set COMPLEX_TO_PREVIEW to one of:", [a.complex_label for a in results if a])


In [ ]:
#@title B5. Download all batch results as a zip { display-mode: "form" }
import shutil
zip_path = shutil.make_archive("/content/batch_results", "zip", batch_out)
from google.colab import files
files.download(zip_path)
